# 02 — Pré-processamento

Cada decisão precisa de justificativa escrita. Decidir *não* criar features é
aceitável, desde que o motivo esteja explícito.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42
RAW = Path("..") / "data" / "raw"
APPLICATION_FILE = RAW / "application_record.csv"
CREDIT_FILE = RAW / "credit_record.csv"
PROCESSED = Path("..") / "data" / "processed"
TARGET = "TARGET"

pd.set_option("display.max_columns", None)

In [2]:
app_df = pd.read_csv(APPLICATION_FILE)
credit_df = pd.read_csv(CREDIT_FILE)
print(f"Aplicações: {app_df.shape}")
print(f"Histórico de crédito: {credit_df.shape}")

Aplicações: (438557, 18)
Histórico de crédito: (1048575, 3)


## 1. Qualidade e dados faltantes

Primeiro verifico valores ausentes e duplicidade. O histórico tem várias linhas por cliente por desenho; aplicações duplicadas por `ID`, por outro lado, são ambíguas e serão removidas integralmente.


In [3]:
nulos = app_df.isna().sum()
display(pd.DataFrame({"nulos": nulos, "pct": (nulos / len(app_df) * 100).round(2)}).query("nulos > 0").sort_values("nulos", ascending=False))
print("IDs de aplicação duplicados:", app_df["ID"].duplicated().sum())
print("Ocupações ausentes:", app_df["OCCUPATION_TYPE"].isna().sum())

,nulos,pct
OCCUPATION_TYPE,134203,30.6


IDs de aplicação duplicados: 47
Ocupações ausentes: 134203


**Decisão:** registros de aplicação com `ID` repetido serão excluídos para não escolher arbitrariamente um perfil. Valores ausentes de ocupação viram a categoria `Outros`; a ausência de tempo de emprego será imputada dentro do pipeline de modelagem, sem usar informação do teste.


## 2. Definição da variável alvo

Um cliente será classificado como mau pagador (`TARGET = 1`) se houver ao menos um mês com atraso de 60 dias ou mais (status 2, 3, 4 ou 5). Os status C, X, 0 e 1 ficam na classe 0.


In [4]:
credit_df["MAU_PAGADOR"] = credit_df["STATUS"].astype(str).isin(["2", "3", "4", "5"])
target_df = credit_df.groupby("ID", as_index=False)["MAU_PAGADOR"].max()
target_df["TARGET"] = target_df.pop("MAU_PAGADOR").astype(int)

app_df = app_df.loc[~app_df["ID"].duplicated(keep=False)].copy()
df = app_df.merge(target_df, on="ID", how="inner")
print("Distribuição do alvo:")
display(df["TARGET"].value_counts().rename_axis("TARGET").to_frame("clientes"))
print("Base após o merge:", df.shape)

Distribuição do alvo:


,clientes
TARGET,
0,35841
1,616


Base após o merge: (36457, 19)


## 3. Normalização / padronização

A escala não é ajustada aqui. No notebook 03, `StandardScaler` fica dentro de um `Pipeline` e é treinado apenas em cada fold de treino. Assim evitamos vazamento do conjunto de teste.


In [5]:
print("A padronização será aplicada no pipeline do notebook 03, após o split.")


A padronização será aplicada no pipeline do notebook 03, após o split.


## 4. Feature engineering

Idade e tempo de emprego são convertidos de dias para anos. O sentinela `365243` de `DAYS_EMPLOYED` vira nulo. Ocupação ausente ou rara é agrupada em `Outros`; valores extremos de filhos e tamanho da família são limitados a 5 e 7, respectivamente. O indicador de celular é removido por ser constante.


In [6]:
df["OCCUPATION_TYPE"] = df["OCCUPATION_TYPE"].fillna("Outros")
df["DAYS_BIRTH"] = df["DAYS_BIRTH"].abs() / 365.25
df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan).abs() / 365.25
df["CNT_CHILDREN"] = df["CNT_CHILDREN"].clip(upper=5)
df["CNT_FAM_MEMBERS"] = df["CNT_FAM_MEMBERS"].clip(upper=7)

frequencia_ocupacao = df["OCCUPATION_TYPE"].value_counts(normalize=True)
ocupacoes_principais = frequencia_ocupacao[frequencia_ocupacao >= 0.03].index
df["OCCUPATION_TYPE"] = df["OCCUPATION_TYPE"].where(
    df["OCCUPATION_TYPE"].isin(ocupacoes_principais), "Outros"
)
if df["FLAG_MOBIL"].nunique(dropna=False) == 1:
    df = df.drop(columns="FLAG_MOBIL")

display(df.head())
print("Nulos restantes:", int(df.isna().sum().sum()))

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,TARGET
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,32.867899,12.435318,1,0,0,Outros,2.0,0
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,32.867899,12.435318,1,0,0,Outros,2.0,0
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,58.792608,3.104723,0,0,0,Outros,2.0,0
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,52.320329,8.353183,0,1,1,Sales staff,1.0,0
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,52.320329,8.353183,0,1,1,Sales staff,1.0,0


Nulos restantes: 6135


## 5. Salvar dataset tratado

O CSV mantém as categorias em formato legível. Codificação, imputação e escala ficam no pipeline do notebook 03 para que sejam ajustadas apenas nos dados de treino.


In [7]:
PROCESSED.mkdir(parents=True, exist_ok=True)
output_path = PROCESSED / "dataset_tratado.csv"
df.to_csv(output_path, index=False)
print(f"Dataset salvo em {output_path} ({df.shape[0]} linhas, {df.shape[1]} colunas).")

Dataset salvo em ..\data\processed\dataset_tratado.csv (36457 linhas, 18 colunas).
